# AFIB classification under noise: evaluation plan and checks

**Aaditya Pathak Â· Siwen Wang Â· Jiayi Huang**  
COMP90051 Statistical Machine Learning

This notebook evaluates saved predictions from the planned patient-separated experiment. Saved logistic regression and TabM runs are complete. HGB now includes both original fold metrics and retuned depth-7/8/9 metrics and prediction archives. The overview below compares their saved summaries. The original full-schema evaluation cells are retained for an optional unified prediction export; they run only when that export and external folds are present.

The question is whether cross-lead feature construction and training augmentation reduce the loss in AFIB classification performance under missing or perturbed extracted ECG measurements. A low stress-related drop is useful only alongside good absolute performance on observed and stressed records. The task is prediction of the existing AFIB annotation, not confirmation of a diagnosis or clinical readiness.

The metric functions are written in `src/afib/evaluation.py`. They follow Lecture 15, slides 7â€“14 and 26â€“32, and the manual-evaluation requirement in Section 2.g. No library metrics, cross-validation splitters or search helpers are used.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
if not (PROJECT / 'src' / 'afib' / 'evaluation.py').exists():
    raise FileNotFoundError('Run from the sml_project package or its notebooks folder.')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.afib.evaluation import (
    binary_metrics, choose_sensitivity_threshold, majority_baseline,
    assert_patient_disjoint, quality_subgroup_metrics, summarise_paired_drops,
    source_quality_groups, validate_scenario_repeats, validate_experiment_arms,
)

from src.afib.preprocessing import SCENARIOS
from src.afib.paths import processed_directory

PREDICTIONS = PROJECT / 'results' / 'comparison' / 'predictions'
INNER_PATH = PREDICTIONS / 'inner_oof_predictions.csv'
OUTER_PATH = PREDICTIONS / 'outer_predictions.csv'
FOLDS_PATH = processed_directory(PROJECT) / 'folds.csv'
READY = INNER_PATH.exists() and OUTER_PATH.exists() and FOLDS_PATH.exists()
print('Ready to validate saved predictions.' if READY else
      'Unified prediction export not present; retained full-schema checks are skipped. '
      'Completed model results are shown below and in notebooks 04–06.')


Unified prediction export not present; retained full-schema checks are skipped. Completed model results are shown below and in notebooks 04–06.


## Saved results side by side

The fixed 0.50 threshold is used throughout. The HGB searches are shown separately so a changed tuning grid is not confused with a feature or augmentation effect.


In [2]:
# Use aggregate fold summaries; datasets and model training are not needed here.
comparison_rows = []
for model_name, label in [('logistic_regression', 'Logistic regression'), ('tabm', 'TabM')]:
    summary = pd.read_csv(PROJECT / 'results' / model_name / 'output_data/summary.csv')
    selected = summary.loc[summary.threshold_policy.eq('fixed_0.50')
                           & summary.variant.isin(list('ABCD'))
                           & summary.scenario.eq('observed')
                           & summary.metric.isin(['macro_f1', 'sensitivity', 'specificity'])]
    for row in selected.itertuples():
        comparison_rows.append({'model': label, 'arm': row.variant, 'metric': row.metric,
                                'mean': row.mean, 'sd': row.sd})
hgb_root = PROJECT / 'results/histogram_gradient_boosting'
for filename, label in [('final_all_metrics.csv', 'HGB original'),
                        ('retuned7_8_9/HGB_ABCD_all_metrics.csv', 'HGB retuned 7/8/9')]:
    metrics = pd.read_csv(hgb_root / filename)
    observed = metrics.loc[metrics.scenario.eq('observed') & metrics.repeat.eq(0)]
    for arm, group in observed.groupby('arm'):
        for metric in ['macro_f1', 'sensitivity', 'specificity']:
            comparison_rows.append({'model': label, 'arm': arm, 'metric': metric,
                                    'mean': group[metric].mean(), 'sd': group[metric].std(ddof=1)})
comparison = pd.DataFrame(comparison_rows)
comparison['mean (SD)'] = comparison.apply(lambda row: f"{row['mean']:.4f} ({row['sd']:.4f})", axis=1)
display(comparison.pivot(index=['arm', 'metric'], columns='model', values='mean (SD)'))
print('SD is variation across ten outer folds. Retuned HGB is a separate later search; interpret its improvement as exploratory.')


SD is variation across ten outer folds. Retuned HGB is a separate later search; interpret its improvement as exploratory.


model               HGB original  ...             TabM
arm metric                        ...                 
A   macro_f1     0.9213 (0.0158)  ...  0.9321 (0.0126)
    sensitivity  0.9514 (0.0243)  ...  0.9108 (0.0258)
    specificity  0.9795 (0.0035)  ...  0.9871 (0.0039)
B   macro_f1     0.9230 (0.0143)  ...  0.9389 (0.0137)
    sensitivity  0.9496 (0.0198)  ...  0.9199 (0.0291)
    specificity  0.9803 (0.0037)  ...  0.9884 (0.0028)
C   macro_f1     0.9156 (0.0122)  ...  0.9338 (0.0121)
    sensitivity  0.9549 (0.0209)  ...  0.9134 (0.0281)
    specificity  0.9771 (0.0034)  ...  0.9874 (0.0035)
D   macro_f1     0.9179 (0.0126)  ...  0.9395 (0.0122)
    sensitivity  0.9583 (0.0190)  ...  0.9166 (0.0261)
    specificity  0.9776 (0.0031)  ...  0.9889 (0.0030)

[12 rows x 4 columns]

## 1. Prediction files and separation checks

Both CSV files need `model`, `feature_set`, `training_arm`, `outer_fold`, `ecg_id`, `patient_id`, `y_true` and `score`. Scores are probabilities of AFIB between 0 and 1. `model` must be one of `logistic_regression`, `hist_gradient_boosting` or `tabm`; `feature_set` is `original` or `enhanced`; `training_arm` is `observed` or `augmented`. A full experiment requires all 12 combinations, each covering outer folds 0 to 9. A partial pilot is not accepted as a complete evaluation. The four feature/training combinations correspond to A = original/observed, B = enhanced/observed, C = original/augmented and D = enhanced/augmented.

The inner file also needs `inner_fold`. For each selected configuration within an outer-training partition, store exactly one held-out inner prediction per training record. These predictions must come from models, augmentation, feature processing and early-stopping decisions fitted without that record's inner-validation patients. Do not replace them with in-sample scores or scores from the outer test fold. The checks below verify the saved split membership; they cannot establish how an external model was fitted.

The outer file also needs `scenario` and `repeat`. Use `observed` for the reference. Save one row per record per stress repeat, including every outer-held-out record. The full protocol is imported from `SCENARIOS`: `observed` at repeat 0, and every stress condition at repeats 0, 1 and 2. A condition missing uniformly from all models is still incomplete.

Quality groups are derived directly from external `reference/ptbxl_database.csv`: any non-empty baseline-drift, static-noise, burst-noise or electrode-problem annotation gives `recorded_quality_flag`; otherwise the group is `no_recorded_quality_flag`. Absence of a recorded flag does not establish a clean signal. Any supplied `quality_group` must match this rule. These annotations are for post-prediction reporting only, not predictor inputs, split selection, corruption assignment or threshold selection.

Use the same saved folds and perturbation seeds for all models and experimental arms. Finish model/feature/scenario selection before evaluating outer predictions. The supplied fold file contains all repeated recordings grouped by patient.


In [3]:
ARM = ['model', 'feature_set', 'training_arm']
GROUP = ARM + ['outer_fold']
if READY:
    folds = pd.read_csv(FOLDS_PATH).set_index('ecg_id')
    inner = pd.read_csv(INNER_PATH)
    outer = pd.read_csv(OUTER_PATH)
    required = set(GROUP + ['ecg_id', 'patient_id', 'y_true', 'score'])
    if not required.union({'inner_fold'}).issubset(inner.columns):
        raise ValueError('The inner OOF table does not match the documented schema.')
    if not required.union({'scenario', 'repeat'}).issubset(outer.columns):
        raise ValueError('The outer table does not match the documented schema.')
    # A complete run needs the prespecified 3 x 2 x 2 experiment in ten folds.
    validate_experiment_arms(inner)
    validate_experiment_arms(outer)
    if not folds.index.is_unique or folds.groupby('patient_id').outer_fold.nunique().max() != 1:
        raise ValueError('Fold records must be unique and patients must stay in one outer fold.')
    if set(folds.outer_fold) != set(range(10)):
        raise ValueError('This protocol requires outer folds 0 to 9.')
    for frame in (inner, outer):
        if frame[list(required)].isna().any().any():
            raise ValueError('Prediction identifiers, labels and scores must not be missing.')
        if not frame.ecg_id.isin(folds.index).all():
            raise ValueError('Prediction ECG IDs are absent from the saved folds.')
        saved = folds.loc[frame.ecg_id]
        if not np.array_equal(saved.patient_id.to_numpy(), frame.patient_id.to_numpy()):
            raise ValueError('Patient IDs do not match the saved split.')
        if not np.array_equal(saved.target_afib.to_numpy(), frame.y_true.to_numpy()):
            raise ValueError('Targets do not match the saved annotations.')
        binary_metrics(frame.y_true, frame.score)  # Validate labels and probability range.
    if inner.duplicated(GROUP + ['ecg_id']).any():
        raise ValueError('Each selected inner OOF record needs exactly one prediction.')
    if outer.duplicated(GROUP + ['scenario', 'repeat', 'ecg_id']).any():
        raise ValueError('Duplicate outer predictions would overcount records.')
    if outer[['scenario', 'repeat']].isna().any().any():
        raise ValueError('Scenario and repeat identifiers must be present.')
    if set(map(tuple, inner[GROUP].drop_duplicates().to_numpy())) != set(map(tuple, outer[GROUP].drop_duplicates().to_numpy())):
        raise ValueError('Inner and outer tables must have matching model/arm/fold groups.')
    for keys, frame in inner.groupby(GROUP, sort=True):
        fold = int(keys[-1])
        expected = folds.index[folds.outer_fold.ne(fold)]
        if set(frame.ecg_id) != set(expected):
            raise ValueError('Inner OOF predictions must cover exactly the outer-training records.')
        expected_inner = folds.loc[frame.ecg_id, f'inner_fold_when_outer_{fold}'].to_numpy()
        if not np.array_equal(expected_inner, frame.inner_fold.to_numpy()):
            raise ValueError('Inner fold IDs do not match the saved patient split.')
        if set(frame.inner_fold) != {0, 1, 2}:
            raise ValueError('All three inner folds must contribute predictions.')
        assert_patient_disjoint(frame.patient_id, folds.loc[folds.outer_fold.eq(fold), 'patient_id'])
    for keys, frame in outer.groupby(GROUP, sort=True):
        fold = int(keys[-1])
        expected = set(folds.index[folds.outer_fold.eq(fold)])
        # Check the declared full protocol, not the first group's observed contents.
        validate_scenario_repeats(frame, SCENARIOS, repeats=(0, 1, 2))
        for _, condition in frame.groupby(['scenario', 'repeat']):
            if set(condition.ecg_id) != expected:
                raise ValueError('Every condition must cover exactly the outer-held-out records.')
        assert_patient_disjoint(folds.loc[folds.outer_fold.ne(fold), 'patient_id'], frame.patient_id)
    for _, frame in outer.groupby(ARM):
        if set(frame.outer_fold) != set(range(10)):
            raise ValueError('Every model/arm requires all ten outer folds.')
    # Quality groups are derived from source annotations only. Never use errors,
    # AFIB labels, synthetic corruption state or scores to define these groups.
    quality_path = processed_directory(PROJECT) / 'metadata.csv'
    if quality_path.exists():
        quality = source_quality_groups(pd.read_csv(quality_path))
        derived_quality = outer.ecg_id.map(quality)
        if derived_quality.isna().any():
            raise ValueError('Some prediction records lack source quality metadata.')
        if 'quality_group' in outer and not outer.quality_group.equals(derived_quality.rename('quality_group')):
            raise ValueError('Supplied quality groups do not match the fixed source-annotation rule.')
        outer['quality_group'] = derived_quality
    elif 'quality_group' in outer:
        raise ValueError('Source metadata is needed to verify supplied quality groups.')
    print('Saved labels, patient separation, full scenario/repeat protocol and source quality groups checked.')


## 2. Choose the operating threshold inside training data

For each outer fold and experimental arm, choose a threshold using the selected configuration's inner out-of-fold scores. The exploratory target is sensitivity of at least 0.90 on these inner scores. Among eligible thresholds, maximise specificity, then sensitivity, then the threshold itself. The 0.90 value is a study choice, not a medical standard or a promised test sensitivity.

Freeze this threshold for observed and stressed outer records. Report sensitivity and specificity together, with precision (positive predictive value) and NPV. Also report a fixed 0.5 threshold as a reproducible reference. Labels without AFIB include other heart conditions. Do not call them healthy controls. Scores from class weighting or augmentation may be miscalibrated: measure Brier score, and avoid interpreting probabilities as calibrated risks without evidence.


In [4]:
if READY:
    selected = []
    for keys, frame in inner.groupby(GROUP, sort=True):
        result = choose_sensitivity_threshold(frame.y_true, frame.score, .90)
        if result['status'] != 'selected_on_inner_oof':
            raise ValueError('An inner partition has only one class; the operating threshold is unavailable.')
        selected.append(dict(zip(GROUP, keys)) | result)
    thresholds = pd.DataFrame(selected)
    outer = outer.merge(thresholds[GROUP + ['threshold']], on=GROUP, validate='many_to_one')
    print('Thresholds selected from inner OOF scores and attached to outer records.')


## 3. Absolute performance, quality groups and a majority baseline

Calculate macro-F1, balanced accuracy, sensitivity, specificity, precision, NPV, AUROC, average precision (AP) and Brier score ourselves. AP groups tied scores at the same threshold; it is not trapezoidal PR area. Undefined denominators are `NaN`, which should appear as NA in the report.

A majority classifier can have high accuracy when AFIB is uncommon. Choose its label and prevalence from each outer-training partition and report it alongside the three models as a sanity check, not a fourth fitted algorithm. Repeated records can influence record-level results, so discuss patient-level resampling or a prespecified one-record-per-patient sensitivity analysis.

Quality groups are descriptive. Show their sample sizes and class counts. A group with only one class has no meaningful two-class comparison; its AUROC, AP, macro-F1 and balanced accuracy are NA. Remaining defined rates and confusion counts are retained. Subgroup results do not replace the main paired analysis and do not establish external validity.


In [5]:
if READY:
    metric_rows, subgroup_rows = [], []
    CONDITION = GROUP + ['scenario', 'repeat']
    for keys, frame in outer.groupby(CONDITION, sort=True):
        identity = dict(zip(CONDITION, keys))
        for policy, threshold in [('inner_sensitivity_0.90', float(frame.threshold.iloc[0])), ('fixed_0.50', .5)]:
            result = binary_metrics(frame.y_true, frame.score, threshold)
            metric_rows.append(identity | {'threshold_policy': policy} | result)
            if 'quality_group' in frame:
                groups = quality_subgroup_metrics(frame.y_true, frame.score, frame.quality_group, threshold)
                subgroup_rows.extend(identity | {'threshold_policy': policy} | row for row in groups.to_dict('records'))
    metrics = pd.DataFrame(metric_rows)
    baseline_rows = []
    for fold in range(10):
        train = folds.loc[folds.outer_fold.ne(fold), 'target_afib']
        test = folds.loc[folds.outer_fold.eq(fold), 'target_afib']
        baseline_rows.append({'outer_fold': fold} | majority_baseline(train, test))
    majority = pd.DataFrame(baseline_rows)
    print(metrics.shape[0], 'fold/condition metric rows calculated; majority baseline calculated for ten folds.')


## 4. Paired changes under stress

Average each stress condition's perturbation repeats inside an outer fold first. Subtract that fold's stressed score from its observed score; for Brier score reverse the subtraction so positive change always means deterioration. Average these ten paired changes and use sample standard deviation across the ten folds as descriptive error bars. Do not count perturbation repeats, augmented copies or multiple recordings as independent patients.

The outer folds share training records, so these SD bars are not confidence intervals and do not justify an independent-fold significance test. A patient-cluster bootstrap of saved out-of-fold predictions may be added with clearly stated conditional uncertainty, but would still not capture full model-retraining uncertainty. Report absolute scores as well as drops, including observed-performance costs of augmentation. A method with poor observed performance can otherwise appear robust simply because there is little performance left to lose.

For the final four-page report, prioritise the question: paired robustness curves, sensitivity/specificity and AP, then quality groups and Brier score where space allows. Epoch-loss plots are not an accepted experimental result under Section 2.g. Sample-size learning curves are a different result.


In [6]:
if READY:
    metrics_to_report = ['macro_f1', 'balanced_accuracy', 'sensitivity', 'specificity',
                         'precision', 'npv', 'accuracy', 'auroc', 'average_precision', 'brier']
    all_effects, all_summaries = [], []
    for keys, frame in metrics.groupby(ARM + ['threshold_policy'], sort=True):
        identity = dict(zip(ARM + ['threshold_policy'], keys))
        for metric in metrics_to_report:
            effects, summary = summarise_paired_drops(frame, metric, higher_is_better=(metric != 'brier'))
            all_effects.extend(identity | row for row in effects.to_dict('records'))
            all_summaries.extend(identity | row for row in summary.to_dict('records'))
    paired_effects = pd.DataFrame(all_effects)
    paired_summary = pd.DataFrame(all_summaries)
    OUTPUT = PROJECT / 'results' / 'comparison' / 'evaluation'
    OUTPUT.mkdir(parents=True, exist_ok=True)
    thresholds.to_csv(OUTPUT / 'inner_selected_thresholds.csv', index=False)
    metrics.to_csv(OUTPUT / 'outer_metrics.csv', index=False)
    majority.to_csv(OUTPUT / 'majority_baseline.csv', index=False)
    paired_effects.to_csv(OUTPUT / 'paired_fold_deterioration.csv', index=False)
    paired_summary.to_csv(OUTPUT / 'paired_summary.csv', index=False)
    if subgroup_rows:
        pd.DataFrame(subgroup_rows).to_csv(OUTPUT / 'quality_subgroups.csv', index=False)
    display(paired_summary)
else:
    print('Optional unified-export evaluation skipped; use the per-model check_results.py scripts for the saved bundles.')


Optional unified-export evaluation skipped; use the per-model check_results.py scripts for the saved bundles.


## Checks before interpreting results

Run `python -m unittest discover -s tests -v` from the package folder. Evaluation tests cover confusion counts, tied AUROC/AP, missing classes, undefined rates, training-only threshold selection, threshold freezing, majority-label selection, patient overlap and averaging repeats before fold-level error bars.

The code checks arithmetic and data contracts. It does not show that a classifier is medically safe, that a synthetic noise level matches a device, or that a feature extractor behaves the same way under raw waveform noise. These remain limitations of the feature-level study.


The LR and TabM result folders additionally contain patient-weighted metrics, calibration reliability tables, controls, and paired method differences. These are produced by their model-specific reporting modules. The retained unified-export scaffold above computes record-level metrics and deterioration within each arm; it does not add sample-size learning curves or patient-cluster uncertainty.


## Keep the AFIB goal ahead of reconstruction

For the separate median-versus-ridge comparison, keep original features, observed-only logistic regression, patient folds, class weights, masks and classifier search matched. Learn replacements for the 216 non-P continuous fields only; the other fields retain baseline handling. Keep these extra predictions separate from the main twelve model/arm combinations.

Observed inputs and three missing leads are the primary imputation comparisons; nominal 10% cell masking and one/six hidden leads describe sensitivity. Use three paired repeats. Macro-F1 and AFIB sensitivity are primary outcomes, with specificity, false alarms and unweighted Brier/reliability as required trade-off checks. Any calibration uses inner out-of-fold predictions only and remains frozen under stress.

Report recovery-only gains, prediction gains, trade-offs, harm or inconclusive results explicitly. Do not infer AFIB benefit from lower MAE, artificial masks from natural gaps, or calibrated risks from class weighting. The course sets validation requirements, not a medical sensitivity, specificity or MAE pass threshold. The completed LR ridge comparison is available in notebook 04 and its saved result tables.
